# Inference Pipeline for VIPER Model (Based on SWIFT)

> **Note**: This notebook documents the inference procedure used to generate predictions and token-level log probabilities.
> The model was fine-tuned on a constrained output format: each prediction consists of exactly **two tokens** —
> (1) the label (`"positive"` or `"negative"`), and (2) an end-of-sequence token.

## Example: VIPER Inference Statistics
- **Test set**: 205 clinical samples → ~250,000 variant records
- **Hardware**: 1× NVIDIA A100 (40GB) GPU
- **Peak GPU memory**: ~37GB
- **Inference time**: < 4 hours

## Key Requirement
To extract **token-level log probabilities** (`--logprobs true`), we patched three files in the `swift` library:
- `llm/infer/infer.py`
- `llm/infer/protocol.py`
- `llm/infer/infer_engine/vllm_engine.py`

> These modifications enable the `vLLM` backend to return joint probabilities over the two-token output sequence.

In [ ]:
# Locate SWIFT installation path
import swift, os
print(f"SWIFT version: {swift.__version__}")
print(f"SWIFT root: {os.path.dirname(swift.__file__)}")

## Step 1: Patch SWIFT Source Files

The official `swift` does not propagate `logprobs` from `vLLM` to the final output when using `swift infer`.
Our patched versions ensure that:
- The `logprobs` and `prop` fields are included in each inference result
- The probability of the **label token** (first generated token) is correctly extracted

> This modifies your current environment’s `site-packages`. Use with caution.

In [ ]:
# Apply patches to SWIFT source files
import shutil

SWIFT_ROOT = os.path.dirname(swift.__file__)
PATCH_DIR = "./patched_swift_files"

files_to_patch = [
    "llm/infer/infer.py",
    "llm/infer/protocol.py",
    "llm/infer/infer_engine/vllm_engine.py"
]

for rel_path in files_to_patch:
    src = os.path.join(PATCH_DIR, os.path.basename(rel_path))
    dst = os.path.join(SWIFT_ROOT, rel_path)

    if not os.path.exists(src):
        raise FileNotFoundError(f"Patched file missing: {src}")
    os.makedirs(os.path.dirname(dst), exist_ok=True)

    print(f"Patching: {dst}")
    shutil.copy2(src, dst)

print("SWIFT patched for logprob support.")

## Step 2: Run Inference with `swift infer`

We use the `vLLM` backend for fast, batched inference with log probability extraction.

> Output is saved as a JSONL file at:
> `{ckpt_dir}/infer_result/{timestamp}.jsonl`

In [ ]:
# Display inference command
inference_script = r'''
export nproc_per_node=1
export NPROC_PER_NODE=$nproc_per_node
export CUDA_VISIBLE_DEVICES=1
export MASTER_PORT=29885

ray stop
ray start --head

swift infer \
    --ckpt_dir {check_point} \
    --model_type qwen3 \
    --attn_impl flash_attn \
    --infer_backend vllm \
    --val_dataset {testset_dir}/*.jsonl \
    --max_model_len 8192 \
    --max_new_tokens 2 \
    --gpu_memory_utilization 0.9 \
    --tensor_parallel_size 1 \
    --temperature 0 \
    --logprobs true \
    --top_p 1 \
    --top_k -1

ray stop
'''

print(inference_script)

In [ ]:
# Uncomment to run inference (after filling paths)
# CHECKPOINT = "/path/to/checkpoint"
# TESTSET_DIR = "/path/to/test/jsonls"

# !CUDA_VISIBLE_DEVICES=1 ray start --head
# !swift infer \
#     --ckpt_dir $CHECKPOINT \
#     --model_type qwen3 \
#     --infer_backend vllm \
#     --val_dataset $TESTSET_DIR/*.jsonl \
#     --max_new_tokens 2 \
#     --logprobs true \
#     --temperature 0 \
#     --gpu_memory_utilization 0.9
# !ray stop

## Step 3: Post-process Inference Results

The raw output is a JSONL file where each line contains:
- `response`: predicted label (`"positive"` / `"negative"`)
- `prop`: list of log probabilities for each generated token

We parse this into structured Excel files per sample using a mapping file (`sample_records.csv`) that specifies how many variants belong to each sample.

In [3]:
import pandas as pd
records = pd.read_csv("sample_records.csv")
records.head(3)

sample_name,line_count
1,462
2,435


In [ ]:
# Parsing utilities for inference results
import json
import demjson3
import os

def parse_jsonl_to_df(file_path, start_line, end_line):
    """
    Parse a segment of the inference JSONL file into a DataFrame.
    Each record is expected to contain nested 'labels' and 'anno' fields.
    """
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for i in range(start_line - 1, min(end_line, len(lines))):
            line = lines[i].strip()
            if not line:
                continue
            try:
                record = demjson3.decode(line)
            except demjson3.JSONDecodeError as e:
                print(f"JSON decode error at line {i+1}: {e}")
                continue

            # Safely extract nested fields
            label_str = record.get('labels', '{}')
            label = {}
            if isinstance(label_str, str):
                try:
                    label = demjson3.decode(label_str)
                except:
                    pass
            elif isinstance(label_str, dict):
                label = label_str

            anno_str = label.get('anno', '{}')
            anno = {}
            if isinstance(anno_str, str):
                try:
                    anno = demjson3.decode(anno_str)
                except:
                    pass
            elif isinstance(anno_str, dict):
                anno = anno_str

            row_data = {
                'cpra': label.get('cpra', ''),
                'gene symbol': label.get('gene symbol', ''),
                'transcript': label.get('transcript', ''),
                'dna change': label.get('dna change', ''),
                'protein change': label.get('protein change', ''),
                'functional consequences': anno.get('functional consequences', ''),
                'Clinvar database': anno.get('Clinvar database', ''),
                'HGMD database': anno.get('HGMD database', ''),
                'SIFT': anno.get('SIFT', ''),
                'Polyphen2-HDIV': anno.get('Polyphen2-HDIV', ''),
                'Polyphen2-HVAR': anno.get('Polyphen2-HVAR', ''),
                'MutationTaster2': anno.get('MutationTaster2', ''),
                'Revel': anno.get('Revel', ''),
                'Condel': anno.get('Condel', ''),
                'SpliceAI': anno.get('SpliceAI', ''),
                'dbscSnv_RF': anno.get('dbscSnv_RF', ''),
                'dbscSnv_ADA': anno.get('dbscSnv_ADA', ''),
                'PhyloP Vertebrates': anno.get('PhyloP Vertebrates', ''),
                'PhyloP Placental Mammals': anno.get('PhyloP Placental Mammals', ''),
                'GERP++': anno.get('GERP++', ''),
                'zygosity': anno.get('zygosity', ''),
                'associated diseases': str(anno.get('associated diseases', '')),
                '1000G': anno.get('1000G', ''),
                'GnomAD': anno.get('GnomAD', ''),
                'GnomAD-EAS': anno.get('GnomAD-EAS', ''),
                'ACMG evidence items': anno.get('ACMG evidence items', ''),
                'pathogenicity assessment': anno.get('pathogenicity assessment', ''),
                'predict': record.get('response', ''),
                'score': record.get('prop', [None])[0]
            }
            data.append(row_data)

    return pd.DataFrame(data)

def write_df_to_excel(df, output_file):
    """Write DataFrame to Excel (overwrite if exists)."""
    if os.path.exists(output_file):
        os.remove(output_file)
    df.to_excel(output_file, index=False, engine='openpyxl')

def process_samples(input_jsonl, sample_ranges_csv, output_dir="./results"):
    """
    Split the full JSONL into per-sample Excel files using sample_ranges_csv.
    """
    os.makedirs(output_dir, exist_ok=True)
    sample_ranges = pd.read_csv(sample_ranges_csv)

    current_line = 1
    for _, row in sample_ranges.iterrows():
        sample_name = row['sample_name']
        line_count = int(row['line_count'])
        start_line = current_line
        end_line = current_line + line_count - 1

        df = parse_jsonl_to_df(input_jsonl, start_line, end_line)
        output_file = os.path.join(output_dir, f"{sample_name}.xlsx")
        write_df_to_excel(df, output_file)

        print(f"Saved {len(df)} records for {sample_name} to {output_file}")
        current_line += line_count

In [ ]:
# Configure paths
INFERENCE_RESULT_JSONL = "{checkpoint_dir}/infer_result/XXXXX.jsonl"  # ← Replace with actual path
SAMPLE_RANGES_CSV = "sample_records.csv"
OUTPUT_DIR = "./parsed_results"

# Run parsing
process_samples(INFERENCE_RESULT_JSONL, SAMPLE_RANGES_CSV, OUTPUT_DIR)

## Final Output

Each clinical sample is saved as a separate Excel file containing:
- Genomic variant metadata
- Annotation from public databases
- Model prediction (`predict`)
- Probability score (`score`) — used for confidence score output

These files are used for downstream evaluation (See `../Eval/example_metadata.xlsx`).